# Task 3 — Password Cracking with Files using CUDA

**6CS005 High Performance Computing | Arkhan Shimar**

The supplied cudaCrypt arithmetic is retained in a device function. Each GPU thread searches the 67,600 possible two-letter/two-digit passwords for one input line. A ceiling-divided grid covers every line, and the final partial block is guarded. This task uses only the synthetic passwords supplied/generated for this assessment.

Run the cells from top to bottom. Choose **Runtime → Change runtime type → T4 GPU** before running. No Google Drive mount or manual input upload is required. The final cell downloads the real run outputs.

The application is C/CUDA; Python cells only arrange files, validate results, and display images.

## Get the public project files

The exact source revision is pinned so later repository edits cannot silently change these inputs.

In [ ]:
%%bash
set -euo pipefail
mkdir -p /content/hpc_task3
curl --fail --location --retry 3 https://codeload.github.com/ArkhanShimar/HPC/tar.gz/5fca2bacfa3490680292677c7514be49755bd1f1 -o /content/hpc_task3.tar.gz
tar -xzf /content/hpc_task3.tar.gz -C /content/hpc_task3 --strip-components=1
echo "Project revision: 5fca2bacfa3490680292677c7514be49755bd1f1"

In [ ]:
%cd /content/hpc_task3/Task3

## Runtime details

In [ ]:
%%bash
set -euo pipefail
{
date -u
gcc --version | head -1
lscpu | grep -E "Model name|CPU\(s\)|Thread|Core"
nvidia-smi
nvcc --version
} | tee environment.txt

## CUDA source

In [ ]:
%%writefile password_cracking.cu
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <stdint.h>
#include <cuda_runtime.h>

// The arithmetic and boundary rules are preserved from CryptForCuda.c.
// A caller-owned array replaces the original static buffer for thread safety.
__device__ void cudaCrypt(const char *raw, char *encrypted) {
    encrypted[0] = raw[0]+2; encrypted[1] = raw[0]-2; encrypted[2] = raw[0]+1;
    encrypted[3] = raw[1]+3; encrypted[4] = raw[1]-3; encrypted[5] = raw[1]-1;
    encrypted[6] = raw[2]+2; encrypted[7] = raw[2]-2;
    encrypted[8] = raw[3]+4; encrypted[9] = raw[3]-4; encrypted[10] = '\0';
    for (int i = 0; i < 10; ++i) {
        int low = i < 6 ? 97 : 48, high = i < 6 ? 122 : 57;
        if (encrypted[i] > high) encrypted[i] = (encrypted[i]-high)+low;
        else if (encrypted[i] < low) encrypted[i] = (low-encrypted[i])+low;
    }
}
__global__ void recover_passwords(const char *encrypted, char *plain, size_t count) {
    size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= count) return;
    char raw[5], candidate[11]; raw[4] = '\0';
    // One password per thread. The grid grows with the number of input lines.
    for (int code = 0; code < 26*26*100; ++code) {
        raw[0] = 'a' + code / 2600;
        raw[1] = 'a' + (code / 100) % 26;
        raw[2] = '0' + (code / 10) % 10; raw[3] = '0' + code % 10;
        cudaCrypt(raw, candidate);
        int same = 1;
        for (int j = 0; j < 10; ++j) if (candidate[j] != encrypted[i*11+j]) { same = 0; break; }
        if (same) { for (int j = 0; j < 5; ++j) plain[i*5+j] = raw[j]; return; }
    }
}
static int checked(cudaError_t code, const char *action) {
    if (code == cudaSuccess) return 1;
    fprintf(stderr, "%s: %s\n", action, cudaGetErrorString(code)); return 0;
}
static int run_gpu(const char *encrypted, char *plain, size_t count) {
    char *device_input = NULL, *device_output = NULL;
    cudaEvent_t start = NULL, stop = NULL;
    cudaDeviceProp properties;
    int status = 1, threads = 128;
    size_t blocks = (count + threads - 1) / threads;
    float ms = 0;
    if (!checked(cudaGetDeviceProperties(&properties, 0), "GPU properties")) goto cleanup;
    if (blocks > (size_t)properties.maxGridSize[0]) { fprintf(stderr, "Input exceeds the GPU grid limit.\n"); goto cleanup; }
    if (!checked(cudaMalloc((void **)&device_input, count*11), "Allocate encrypted passwords") ||
        !checked(cudaMalloc((void **)&device_output, count*5), "Allocate recovered passwords") ||
        !checked(cudaMemcpy(device_input, encrypted, count*11, cudaMemcpyHostToDevice), "Copy input") ||
        !checked(cudaMemset(device_output, 0, count*5), "Clear output") ||
        !checked(cudaEventCreate(&start), "Create start event") ||
        !checked(cudaEventCreate(&stop), "Create stop event")) goto cleanup;
    if (!checked(cudaEventRecord(start), "Record start")) goto cleanup;
    recover_passwords<<<(unsigned)blocks, threads>>>(device_input, device_output, count);
    if (!checked(cudaGetLastError(), "Launch password kernel") ||
        !checked(cudaEventRecord(stop), "Record stop") ||
        !checked(cudaEventSynchronize(stop), "Wait for kernel") ||
        !checked(cudaEventElapsedTime(&ms, start, stop), "Measure kernel") ||
        !checked(cudaMemcpy(plain, device_output, count*5, cudaMemcpyDeviceToHost), "Copy results")) goto cleanup;
    printf("GPU: %s | Passwords: %zu | Blocks: %zu | Threads/block: %d\n", properties.name, count, blocks, threads);
    printf("Kernel time: %.3f ms\n", ms); status = 0;
cleanup:
    if (start && !checked(cudaEventDestroy(start), "Destroy start event")) status = 1;
    if (stop && !checked(cudaEventDestroy(stop), "Destroy stop event")) status = 1;
    if (device_input && !checked(cudaFree(device_input), "Free input")) status = 1;
    if (device_output && !checked(cudaFree(device_output), "Free output")) status = 1;
    return status;
}
int main(int argc, char **argv) {
    if (argc != 2) { fprintf(stderr, "Usage: %s passwords.txt\n", argv[0]); return 1; }
    FILE *input = fopen(argv[1], "r");
    if (!input) { perror(argv[1]); return 1; }
    size_t count = 0, capacity = 256, missing = 0;
    char *encrypted = (char *)malloc(capacity*11), *plain = NULL;
    char line[32]; FILE *output = NULL; int status = 1;
    if (!encrypted) goto cleanup;
    while (fgets(line, sizeof(line), input)) {
        size_t length = strcspn(line, "\r\n");
        if (length != 10 || (line[length] == '\r' && line[length+1] && line[length+1] != '\n')) {
            fprintf(stderr, "Line %zu: expected 10 encrypted characters.\n", count+1); goto cleanup;
        }
        for (int j = 0; j < 10; ++j) {
            if ((j < 6 && (line[j] < 'a' || line[j] > 'z')) || (j >= 6 && (line[j] < '0' || line[j] > '9'))) {
                fprintf(stderr, "Line %zu: invalid encrypted character.\n", count+1); goto cleanup;
            }
        }
        if (count == capacity) {
            if (capacity > SIZE_MAX / 22) goto cleanup;
            char *larger = (char *)realloc(encrypted, capacity*22);
            if (!larger) goto cleanup;
            encrypted = larger; capacity *= 2;
        }
        memcpy(encrypted+count*11, line, 10); encrypted[count*11+10] = '\0'; count++;
    }
    if (ferror(input) || !count) { fprintf(stderr, "Cannot read passwords or file is empty.\n"); goto cleanup; }
    plain = (char *)calloc(count, 5);
    if (!plain || run_gpu(encrypted, plain, count)) goto cleanup;
    output = fopen("decrypted.txt", "w");
    if (!output) { perror("decrypted.txt"); goto cleanup; }
    for (size_t i = 0; i < count; ++i) {
        if (plain[i*5]) fprintf(output, "%s\n", plain+i*5);
        else { fprintf(output, "NOT_FOUND\n"); missing++; }
    }
    if (ferror(output)) goto cleanup;
    printf("Recovered: %zu | Not found: %zu | Output: decrypted.txt\n", count-missing, missing);
    status = missing ? 2 : 0;
cleanup:
    if (fclose(input)) status = 1;
    if (output && fclose(output)) status = 1;
    free(encrypted); free(plain); return status;
}


## Compile

In [ ]:
%%bash
set -euo pipefail
nvcc -O2 -lineinfo password_cracking.cu -o password_cracking

## Run the main example

In [ ]:
%%bash
set -euo pipefail
{
./password_cracking ../data/passwords.txt
printf "\nFirst 12 recovered passwords\n"
head -12 decrypted.txt
printf "\nOutput line count\n"
wc -l decrypted.txt
} | tee run.log

## Correctness checks and measurements

The checks compare against independent CPU references and exercise invalid input. Five measured repeats follow a warm-up where appropriate. CPU timings cover the computation stated by each program; CUDA event timings cover kernels only, not file access, allocation or transfers. These scopes must not be compared as end-to-end speedups.

In [ ]:
%%bash
set -euo pipefail
python ../tests/verify.py 3 | tee validation.log

## Viva discussion

Explain host versus device memory, __device__ versus __global__, blockIdx.x * blockDim.x + threadIdx.x, ceiling division, boundary guards, why static buffers are unsafe here, and the limitations of one password per thread.

## Save the evidence from this run

This downloads output files and validation logs. To keep notebook output cells too, choose **File → Download → Download .ipynb** after the run. A fresh viewer can rerun the same public notebook.

In [ ]:
from pathlib import Path
import shutil, zipfile
from google.colab import files
archive = Path('/content/Task3_outputs.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for pattern in ['*.txt', '*.log', '*.png', 'validation/*.json']:
        for path in Path('.').glob(pattern):
            z.write(path, path.as_posix())
files.download(str(archive))